# RQ1: Extreme-heat trends in Kanto via a Generalised Additive Model, 1980–2024

**Research question.** How have extreme heat event frequencies, namely extreme-heat days (conventionally ≥35 °C) and tropical nights (≥25 °C overnight), changed in the Kanto Metropolitan Area over 1980–2024?

**Approach.** We estimate the trend in each heat-index series with a **Generalised Additive Model (GAM)**, a method that recovers the shape of a trend from the data rather than assuming it. The analysis works through the full GAM workflow: model formulation, smoothing-parameter selection, the fitted smooth and its goodness-of-fit diagnostics, the derivative of the smooth (the rate of change through time), and bootstrap uncertainty. Non-parametric trend tests (Mann-Kendall, Pettitt, Sen's slope) are kept in a companion notebook to keep the focus here on the GAM; they corroborate the findings below.

## Interpreting the heat-index series
- **Counts are trend indicators, not face-value frequencies.** A station comparison (notebook 03) found that the ERA5-Land area-mean carries a daily-maximum cold bias of roughly 1.7 °C, because averaging over the roughly 9 km grid cell smooths away the local temperature peaks a station thermometer records. The heat-day counts used here are spatial area-means of per-cell counts (fractional) and are suppressed by this smoothing, so they are read as trends rather than magnitudes. The same finding motivates counting extreme-heat days at an adjusted ≥33 °C threshold, explained in the Data section below.
- **Data source:** ERA5-Land Kanto area-mean, with Section 7 validating the trend direction against JMA station observations.
- The urban heat island is the most plausible mechanism for part of the reanalysis-versus-station gap. It is acknowledged here as a limitation rather than quantified.

## Setup

In [1]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats as sp_stats

_p = pathlib.Path.cwd()
while not (_p / ".git").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.climate_utils import (
    annual_threshold_count,
    bootstrap_gam,
    fit_gam,
    gam_derivative,
    kanto_mean,
)
from src.jma_data import load_stations

NOTEBOOK_ID = "04"
NOTEBOOK_NAME = "04_rq1_extreme_heat_gam_v1.1"
FIGURES_DIR = _p / "figures" / NOTEBOOK_NAME
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = _p / "data" / "processed"


def savefig(fig, section, fig_num, title):
    '''Save *fig* to figures/<notebook>/ using [NB]_[section]-[fig#]_[title].png.'''
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_ID}_{section}-{fig_num}_{title}.png",
                bbox_inches="tight", dpi=120)

### Analysis helpers

Three helpers, defined once and reused across the indices:

- **`gam_panel`**: fits a GCV-selected GAM and draws the annual data, the fitted smooth and its confidence band, annotated with the effective degrees of freedom (EDoF), deviance explained, selected λ and AIC. Returns the `GAMResult`.
- **`deriv_panel`**: plots the derivative of the fitted smooth (the year-on-year *rate of change*) with a bootstrap confidence band, and shades and marks the year from which the rate becomes significantly positive.
- **`residual_diagnostics`**: the standard residual plots, namely residuals versus fitted, a normal Q-Q plot, and residuals versus year.

In [2]:
def gam_panel(ax, years, y, label, unit, color="steelblue", show_ci=True):
    '''Fit a GCV-selected GAM and draw data + smooth + CI + diagnostics box. Returns GAMResult.'''
    g = fit_gam(years, y)
    ax.plot(years, y, "o", color=color, ms=3.5, alpha=0.7, label="Annual (ERA5-Land area-mean)")
    ax.plot(g.x, g.fitted, color="firebrick", lw=2.3, label="GAM smooth")
    if show_ci:
        ax.fill_between(g.x, g.ci[:, 0], g.ci[:, 1], color="firebrick", alpha=0.15, label="95% CI")
    annot = (f"EDoF = {g.edof:.2f}\n"
             f"dev. explained = {g.deviance_explained:.2f}\n"
             f"lambda = {g.lam:.3g}\n"
             f"AIC = {g.aic:.1f}")
    ax.text(0.02, 0.97, annot, transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="lightgray"))
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{label} ({unit})")
    ax.legend(fontsize=7.5, loc="lower right")
    return g


def _sustained_sig_onset(x_grid, sig):
    '''Start year of the contiguous significant run that reaches the record's end.

    The derivative's significance is non-contiguous (an early-record epoch, a mid-record
    lull, then a sustained recent rise). We report the onset of the *final* run, the year
    the current acceleration begins, not first-ever significance, which would be captured
    by an edge-adjacent early blip. Returns None if the rate is not significant at the end.
    '''
    if not sig[-1]:
        return None
    i = len(sig) - 1
    while i > 0 and sig[i - 1]:
        i -= 1
    return float(x_grid[i])


def deriv_panel(ax, g, years, y, unit, n_boot=1000, seed=0):
    '''Plot the GAM derivative (rate of change) with bootstrap CI; shade all rate>0 regions
    and mark the onset of the *sustained* significant run.

    We do not mark the derivative's maximum: for these series the rate is still rising at the
    2024 record boundary, so its argmax is pinned to the endpoint and would be a data-edge
    artefact, not a located peak.
    '''
    d = gam_derivative(g.gam, g.x)
    bg = bootstrap_gam(years, y, lam=g.lam, n_boot=n_boot, seed=seed, x_grid=g.x, derivative=True)
    ax.plot(g.x, d, color="firebrick", lw=2.2, label="dGAM/dt (rate)")
    ax.fill_between(g.x, bg["deriv_lo"], bg["deriv_hi"], color="firebrick", alpha=0.15,
                    label="bootstrap 95% CI")
    ax.axhline(0, color="black", lw=0.8)
    y0, y1 = ax.get_ylim()
    sig = bg["deriv_lo"] > 0
    ax.fill_between(g.x, y0, y1, where=sig, color="tab:green", alpha=0.10, label="rate > 0 (95%)")
    ax.set_ylim(y0, y1)
    sig_from = _sustained_sig_onset(g.x, sig)
    if sig_from is not None:
        ax.axvline(sig_from, color="tab:green", ls=":", lw=1.5,
                   label=f"sustained rate > 0 from ≈ {sig_from:.0f}")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"rate ({unit}/yr)")
    ax.legend(fontsize=7.5, loc="upper left")
    return {"rate_sig_from": sig_from}


def residual_diagnostics(g, years, y, label, color="steelblue"):
    '''Three-panel GAM residual diagnostics.'''
    resid = g.residuals
    fitted_obs = np.asarray(y, float) - resid
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.4), constrained_layout=True)
    axes[0].scatter(fitted_obs, resid, color=color, s=14, alpha=0.7)
    axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set_xlabel("Fitted"); axes[0].set_ylabel("Residual")
    axes[0].set_title("Residuals vs Fitted", fontsize=9)
    sp_stats.probplot(resid, plot=axes[1])
    axes[1].set_title("Normal Q-Q", fontsize=9)
    axes[2].plot(years, resid, "o-", color=color, ms=3, lw=0.8)
    axes[2].axhline(0, color="black", lw=0.8)
    axes[2].set_xlabel("Year"); axes[2].set_ylabel("Residual")
    axes[2].set_title("Residuals vs Year", fontsize=9)
    fig.suptitle(f"GAM residual diagnostics: {label}", fontsize=10)
    return fig

## Data: the heat-index series

The analysis rests on four annual series spanning 1980–2024, one value per year, all derived from ERA5-Land 2 m temperature over the Kanto grid with daily aggregates taken in Japan Standard Time. All four series use the same July–September heat-wave window, so the counts and the magnitudes describe the same season and are directly comparable:

- **`tropical_nights_ge25`** and **`hot_days_ge33`**: the annual count of tropical nights (daily minimum ≥25 °C) and extreme-heat days (daily maximum ≥33 °C) within the July–September heat-wave window. Each count is formed per grid cell and then averaged over the Kanto land cells (water cells in the box carry no data and are excluded), so the values are *fractional*, a spatial mean of per-cell counts rather than a single-station tally.
- **`summer_tmin_mean_c`** and **`summer_tmax_mean_c`**: the **seasonal mean of the daily minimum and daily maximum** 2 m temperature. For each day the minimum (or maximum) is taken; these are then averaged over all July–September days in the year and over the Kanto grid. These continuous magnitudes underlie the two threshold counts. The quantity is a seasonal-and-spatial average of the daily extremes, not a single day.

**On the ≥33 °C threshold.** The conventional definition of an extreme-heat day in Japan is a daily maximum of ≥35 °C (the Japan Meteorological Agency's *mōshobi*). That definition is calibrated to point observations, not to a spatial average. ERA5-Land reports the mean over a roughly 9 km grid cell, which smooths away the local peaks a station thermometer records, so a ≥35 °C count on the area-mean systematically under-registers extreme-heat days. The station comparison in notebook 03 quantifies the effect as a daily-maximum cold bias of roughly 1.7 °C. We therefore count extreme-heat days at ≥33 °C on the area-mean series, approximately the standard 35 °C threshold minus that bias. This is a deliberately simple adjustment rather than a precise calibration: the offset also folds in grid-versus-point representativeness and the urban siting of the stations, so the ≥33 °C count, like the other counts, is read for its trend rather than its absolute level.

The series are cached at `data/processed/kanto_annual_heat_indices.csv` (check-then-compute from the ERA5-Land daily and monthly caches via `src.climate_utils`). Their spatial structure, decade-scale maps, and construction are explored in full in the companion exploratory notebooks: notebook 01 for the ERA5-Land fields, and notebook 02 for the JMA station records against which they are validated.

In [3]:
csv_path = PROCESSED_DIR / "kanto_annual_heat_indices.csv"
if csv_path.exists():
    heat = pd.read_csv(csv_path, index_col="year")
else:
    daily_tmax = xr.open_dataarray(PROCESSED_DIR / "daily_max_t2m.nc").load()
    daily_tmin = xr.open_dataarray(PROCESSED_DIR / "daily_min_t2m.nc").load()

    hot = kanto_mean(annual_threshold_count(daily_tmax, 33.0, months=(7, 8, 9)))
    trop = kanto_mean(annual_threshold_count(daily_tmin, 25.0, months=(7, 8, 9)))

    def _summer_area_mean(daily):
        # Mean of the DAILY values over the Jul-Sep heat-wave window, area-averaged
        # over the Kanto land cells -- the same window as the two counts.
        # (daily_max/daily_min are the daily extremes; averaging them gives the
        # seasonal mean daily maximum / minimum -- not the monthly single-hour extreme.)
        jas = daily.sel(valid_time=daily["valid_time"].dt.month.isin([7, 8, 9]))
        ts = kanto_mean(jas).groupby("valid_time.year").mean()
        return pd.Series(ts.values, index=ts["year"].values)

    heat = pd.DataFrame({
        "hot_days_ge33": pd.Series(hot.values, index=hot["valid_time"].dt.year.values),
        "tropical_nights_ge25": pd.Series(trop.values, index=trop["valid_time"].dt.year.values),
        "summer_tmax_mean_c": _summer_area_mean(daily_tmax),
        "summer_tmin_mean_c": _summer_area_mean(daily_tmin),
    }).sort_index()
    heat.index.name = "year"
    heat.round(4).to_csv(csv_path)

years = heat.index.values.astype(float)
heat.head()


,hot_days_ge33,tropical_nights_ge25,summer_tmax_mean_c,summer_tmin_mean_c
year,,,,
1980,0.0000,0.1947,24.6914,19.6739
1981,0.0000,0.9053,26.1239,20.1007
1982,0.0000,0.0684,25.1957,19.5908
1983,0.5684,1.0158,25.9358,20.3407
1984,3.7105,3.3000,27.4202,20.8812


The four series set up the analysis that follows. Both continuous magnitudes rise steadily: the mean daily maximum temperature from about 25.9 °C in the early 1980s to about 28.4 °C in 2020–2024, and the mean daily minimum from about 20.1 to about 22.5 °C, increases of roughly 2.4–2.5 °C over the record. The two threshold counts rise far more sharply in relative terms: the area-mean tropical-night count climbs from about 1.1 nights per year to about 16.3, peaking at 30.0 in 2024, and the hot-day count (≥33 °C) from about 0.9 days per year to about 6.2, peaking near 11.5 in 2023. Whether each rise is steady or accelerating is the question the GAM fits below address.

## 1. What a GAM is, and why it fits this problem

A Generalised Additive Model replaces the linear predictor of a GLM with a sum of **smooth functions** of the covariates. For a single covariate (year $t$) and a Gaussian response, the model is

$$ y_t = \beta_0 + f(t) + \varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0, \sigma^2), $$

where the smooth $f(t)$ is expanded in a **penalised regression-spline basis**, $f(t) = \sum_{k=1}^{K} \beta_k b_k(t)$. The basis functions $b_k$ are fixed; the coefficients $\beta_k$ are estimated by penalised least squares, minimising

$$ \sum_t \big(y_t - \beta_0 - f(t)\big)^2 \; + \; \lambda \int f''(t)^2 \, dt. $$

The **wiggliness penalty** $\lambda \int f''^2$ shrinks curvature: large $\lambda$ drives $f$ toward a straight line, small $\lambda$ lets it interpolate the noise. The basis size $K$ (`n_splines`) only sets an upper bound on flexibility; once $\lambda$ is chosen, the *effective* degrees of freedom (EDoF) of the fit fall well below $K$. Two properties make a GAM well suited to this problem:

- **It does not impose a functional form.** The East Asian Summer Monsoon literature warns that the trend is likely non-monotonic, so a single linear slope (or a fixed-degree polynomial, which is globally sensitive to the endpoints) would prejudge the shape. The GAM lets the data choose the shape, with the penalty guarding against overfitting.
- **Uncertainty is built in.** The spline coefficients carry a covariance, giving a confidence band on $f(t)$ directly, and the smooth is differentiable, so the *rate of change* $f'(t)$ and its uncertainty are available (Section 4).

## 2. Smoothing-parameter selection (the bias–variance trade-off)

$\lambda$ is the one tuning decision that matters. We select it by **generalised cross-validation (GCV)**, which estimates out-of-sample error without an explicit held-out set. We illustrate the mechanics on the tropical-night series, chosen as the worked example because it is the series with the clearest non-linear structure: demonstrating a smoothing trade-off needs a curve that can actually be over- or under-smoothed, which a near-linear series cannot show. The identical procedure selects $\lambda$ independently for each of the four series in Sections 4 and 5, and for the two series without resolvable curvature it correctly settles on a straight line. The left panel shows the tropical-night series fitted at a deliberately *under*-smoothed $\lambda$, an *over*-smoothed $\lambda$, and the GCV-selected value; the right panel plots the GCV score across a $\lambda$ grid, whose minimum is the selected value.

In [4]:
tn = heat["tropical_nights_ge25"].values

lam_grid = np.logspace(-3, 3, 30)
gcv_curve = [fit_gam(years, tn, gridsearch=False, lam=lam).gcv for lam in lam_grid]

sel = fit_gam(years, tn)                                   # GCV gridsearch
under = fit_gam(years, tn, gridsearch=False, lam=1e-3)     # too wiggly
over = fit_gam(years, tn, gridsearch=False, lam=1e4)       # too stiff

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))
axL.plot(years, tn, "o", color="tab:brown", ms=4, alpha=0.7, label="Annual")
axL.plot(under.x, under.fitted, color="tab:blue", lw=1.5, alpha=0.8,
         label=f"under-smoothed (λ=1e-3, EDoF={under.edof:.1f})")
axL.plot(over.x, over.fitted, color="tab:green", lw=1.5, alpha=0.8,
         label=f"over-smoothed (λ=1e4, EDoF={over.edof:.1f})")
axL.plot(sel.x, sel.fitted, color="firebrick", lw=2.5,
         label=f"GCV-selected (λ={sel.lam:.2g}, EDoF={sel.edof:.1f})")
axL.set_xlabel("Year"); axL.set_ylabel("Tropical nights ≥25 °C (nights)")
axL.set_title("Effect of the smoothing parameter λ")
axL.legend(fontsize=8, loc="upper left")

axR.plot(lam_grid, gcv_curve, "-o", color="black", ms=3)
axR.axvline(sel.lam, color="firebrick", ls="--", label=f"selected λ={sel.lam:.2g}")
axR.set_xscale("log"); axR.set_xlabel("λ (log scale)"); axR.set_ylabel("GCV score")
axR.set_title("GCV vs λ (minimum = selected)")
axR.legend(fontsize=8)
savefig(fig, 2, 1, "smoothing_parameter_selection")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\887404269.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Observation.** The under-smoothed fit chases year-to-year noise (high EDoF, no interpretable trend); the over-smoothed fit collapses toward a straight line (EDoF ≈ 2) and erases any non-linearity. The GCV curve is U-shaped, and its minimum picks an intermediate λ whose EDoF describes a smooth but flexible trajectory. This same selection is applied independently to each series below, so each is smoothed only as much as its own data supports.

## 3. Reading a fit: rate of change and its uncertainty

Two further quantities are read from every fitted smooth in the sections below, so both are introduced here once.

**The rate of change (derivative).** Because the smooth $f(t)$ is differentiable, its derivative $f'(t)$ is the instantaneous rate of warming, in units per year, and it is allowed to vary through time in a way a single linear or Sen's slope cannot. Where $f'(t)$ is significantly greater than zero, the series is warming at a statistically resolved rate at that moment. We do not report the year of maximum rate: for a series still rising at the 2024 record boundary the derivative's maximum is pinned to that endpoint and would be a data-edge artefact. Instead we mark the onset of the *sustained* significant run, the contiguous stretch of significantly-positive rate that reaches the end of the record.

**Uncertainty, and why we bootstrap.** A GAM reports an *analytic* confidence band, derived from the fitted coefficients' covariance under normal-theory (Gaussian, large-sample) assumptions. Two features of this problem make those assumptions fragile. First, the sample is small: 45 annual values, from which a flexible smooth has already spent several effective degrees of freedom, leaving few residual degrees of freedom for the large-sample approximation to lean on. Second, the two count series are not Gaussian: their residuals are heteroscedastic with heavy tails (Section 7), directly violating the constant-variance assumption behind the analytic band.

We therefore add a **bootstrap**. We resample the 45 `(year, value)` pairs with replacement, refit the GAM at the selected $\lambda$, and repeat 1000 times; the spread of the resulting curves is an empirical estimate of the fit's sampling variability that assumes no particular error distribution. Where the bootstrap band agrees with the analytic one, the analytic band is validated; where it diverges, the bootstrap is the more trustworthy reading. Two honest limits: the bootstrap does not manufacture information, 45 points remain 45 points, so it sharpens the *reading* of the uncertainty rather than reducing it; and because we resample pairs independently, it does not model year-to-year autocorrelation, which the residuals-versus-year diagnostic (Section 7) is used to check.

The figure below demonstrates the comparison on the tropical-night series, a count, where the Gaussian assumption behind the analytic band is weakest and the bootstrap is therefore the sterner test: the analytic 95% band and the 1000-sample bootstrap band together.

In [5]:
g_tn = fit_gam(years, tn)
boot = bootstrap_gam(years, tn, lam=g_tn.lam, n_boot=1000, seed=2, x_grid=g_tn.x)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(years, tn, "o", color="tab:brown", ms=3.5, alpha=0.7, label="Annual")
ax.plot(g_tn.x, g_tn.fitted, color="firebrick", lw=2.3, label="GAM smooth")
ax.fill_between(g_tn.x, g_tn.ci[:, 0], g_tn.ci[:, 1], color="firebrick", alpha=0.18,
                label="analytic 95% CI")
ax.plot(boot["x"], boot["fit_lo"], color="black", lw=1.0, ls="--", label="bootstrap 95% band")
ax.plot(boot["x"], boot["fit_hi"], color="black", lw=1.0, ls="--")
ax.set_xlabel("Year"); ax.set_ylabel("Tropical nights ≥25 °C (nights)")
ax.set_title("Analytic vs bootstrap uncertainty (tropical nights)")
ax.legend(fontsize=8, loc="upper left")
savefig(fig, 3, 1, "bootstrap_vs_analytic_tropical_nights")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\3880672861.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Observation.** Even for the tropical-night count, where the Gaussian assumption behind the analytic band is weakest, the bootstrap band and the analytic CI largely coincide through the interior of the record, with the bootstrap band running a little wider at the recent, high-count end. Because the case-resampling bootstrap makes no distributional assumption, this near-agreement is reassurance that the analytic band is not badly misleading here, while the slightly wider bootstrap at the end is the expected sign that the analytic interval is mildly optimistic where the count and its variance are largest. Each series below is read with its bootstrap rate band, not the analytic band alone.

## 4. Temperature: mean daily maximum and minimum

The two continuous magnitudes come first: the **mean daily maximum** and **mean daily minimum** 2 m temperature, meaning the daily maxima (or minima) averaged over all July–September days in the year and over the Kanto grid. Each is fitted with the same GCV-selected GAM and read through its fitted smooth (with the analytic confidence band) and its rate of change (with the bootstrap band).

In [6]:
# The four series, each fitted with the identical GCV-selected GAM in the sections below.
specs = [
    ("summer_tmax_mean_c",    "Mean daily maximum temperature (Jul–Sep)", "°C",     "tab:red"),
    ("summer_tmin_mean_c",    "Mean daily minimum temperature (Jul–Sep)", "°C",     "tab:purple"),
    ("hot_days_ge33",         "Hot days ≥33 °C (Jul–Sep)",           "days",   "tab:orange"),
    ("tropical_nights_ge25",  "Tropical nights ≥25 °C (Jul–Sep)",    "nights", "tab:brown"),
]
spec = {s[0]: s for s in specs}
gams, rates = {}, {}

### 4.1 Mean daily maximum temperature

In [7]:
key = "summer_tmax_mean_c"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
gams[key] = gam_panel(ax, years, heat[key].values, label, unit, color=color)
ax.set_title(f"RQ1: {label} GAM fit")
savefig(fig, 4, 1, "tmax_gam_fit")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\860866689.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
key = "summer_tmax_mean_c"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
rates[key] = deriv_panel(ax, gams[key], years, heat[key].values, unit, n_boot=1000, seed=11)
ax.set_title(f"RQ1: {label} rate of change")
savefig(fig, 4, 2, "tmax_gam_rate")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\2628948178.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Mean daily maximum temperature.** The GAM self-selects a straight line (EDoF 2.0, with $\lambda$ at the top of the search range): the daytime mean has warmed at a steady pace, with no detectable change in that pace. The derivative is essentially constant and significantly positive across the whole record (sustained from 1980), at about 0.05 °C per year (0.5 °C per decade). Deviance explained is modest (0.34), reflecting the large year-to-year scatter of a seasonal-mean summary around a slow underlying trend.

### 4.2 Mean daily minimum temperature

In [9]:
key = "summer_tmin_mean_c"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
gams[key] = gam_panel(ax, years, heat[key].values, label, unit, color=color)
ax.set_title(f"RQ1: {label} GAM fit")
savefig(fig, 4, 3, "tmin_gam_fit")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\3770849656.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
key = "summer_tmin_mean_c"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
rates[key] = deriv_panel(ax, gams[key], years, heat[key].values, unit, n_boot=1000, seed=12)
ax.set_title(f"RQ1: {label} rate of change")
savefig(fig, 4, 4, "tmin_gam_rate")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\2907826711.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Mean daily minimum temperature.** The overnight mean behaves differently: here the GAM selects a genuinely non-linear smooth (EDoF 3.8, deviance explained 0.50). The rate of change is positive in the early 1980s, falls to a mid-record lull around 2001 where it is indistinguishable from zero, then climbs: the rate becomes sustained-significantly positive from about 2007 and reaches roughly 0.14 °C per year at the end of the record, nearly three times the daytime rate. Averaged across the whole record the overnight mean warms only slightly faster than the daytime mean (about 0.058 versus 0.051 °C per year), so the important contrast is not the average pace but the shape: daytime warming is steady, overnight warming is accelerating. A diurnal asymmetry has opened in recent decades, and the natural question for the counts is whether they inherit these two shapes.

## 5. Threshold exceedance counts: hot days and tropical nights

The two counts follow: the annual number of hot days (daily maximum ≥33 °C) and tropical nights (daily minimum ≥25 °C) in the July–September window. As set out in the Data section, the ≥33 °C hot-day threshold is the bias-adjusted counterpart of the standard ≥35 °C definition, chosen to offset the peak-smoothing of the area-mean.

### 5.1 Hot days ≥33 °C

In [11]:
key = "hot_days_ge33"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
gams[key] = gam_panel(ax, years, heat[key].values, label, unit, color=color)
ax.set_title(f"RQ1: {label} GAM fit")
savefig(fig, 5, 1, "hot_days_gam_fit")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\2799595179.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
key = "hot_days_ge33"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
rates[key] = deriv_panel(ax, gams[key], years, heat[key].values, unit, n_boot=1000, seed=13)
ax.set_title(f"RQ1: {label} rate of change")
savefig(fig, 5, 2, "hot_days_gam_rate")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\2851828309.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Hot days ≥33 °C.** The count mirrors the daytime magnitude: the GAM again self-selects a straight line (EDoF 2.0, deviance explained 0.19), a steady rise significant across the whole record, from well under one day per year early in the record to around six recently. Its sparsity early on is itself informative: even at the bias-adjusted threshold, area-mean hot days were near-absent in the 1980s, so the signal is a genuine emergence of daytime extremes rather than an intensification of an already-common event.

### 5.2 Tropical nights ≥25 °C

In [13]:
key = "tropical_nights_ge25"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
gams[key] = gam_panel(ax, years, heat[key].values, label, unit, color=color)
ax.set_title(f"RQ1: {label} GAM fit")
savefig(fig, 5, 3, "tropical_nights_gam_fit")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\33082255.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
key = "tropical_nights_ge25"; _, label, unit, color = spec[key]
fig, ax = plt.subplots(figsize=(11, 5.5))
rates[key] = deriv_panel(ax, gams[key], years, heat[key].values, unit, n_boot=1000, seed=14)
ax.set_title(f"RQ1: {label} rate of change")
savefig(fig, 5, 4, "tropical_nights_gam_rate")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\1831552743.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Tropical nights ≥25 °C.** This is the most strongly non-linear of the four (EDoF 4.5, deviance explained 0.62). The smooth is near-flat through the 1990s, then accelerates steeply, reaching about 16 nights per year recently with a single-year peak near 30. Its rate traces the same trajectory as the minimum-temperature magnitude: a mid-record lull around 2000, then a sustained significant acceleration from about 2007, the same onset year as the overnight mean, still rising at the record's end. The night-time count therefore inherits the night-time magnitude's shape, amplified: where the mean gains roughly 0.14 °C per year, the count is growing by nearly two additional nights per year by 2024.

## 6. Validation against JMA station observations

The GAM fits run on the ERA5-Land Kanto area-mean; this section checks their direction against the JMA Tokyo station record. A Kanto-wide area-mean and a single urban station are not the same quantity, so their absolute levels are not expected to match; the test is whether the ERA5-Land and station GAM smooths *rise in parallel*. For hot days the station count uses the standard ≥35 °C threshold while the ERA5-Land count uses the bias-adjusted ≥33 °C, so that panel additionally checks whether the adjusted grid count tracks the standard station count.

In [15]:
_jma = load_stations(["tokyo"]).copy()
_jma["year"] = _jma["date"].dt.year
_jas = _jma[_jma["date"].dt.month.isin([7, 8, 9])]
_yr = list(range(1980, 2025))
jyears = np.array(_yr, dtype=float)

jma_series = {
    "summer_tmax_mean_c":   _jas.groupby("year")["max_temp"].mean().reindex(_yr),
    "summer_tmin_mean_c":   _jas.groupby("year")["min_temp"].mean().reindex(_yr),
    "hot_days_ge33":        _jas[_jas["max_temp"] >= 35.0].groupby("year").size().reindex(_yr, fill_value=0),
    "tropical_nights_ge25": _jas[_jas["min_temp"] >= 25.0].groupby("year").size().reindex(_yr, fill_value=0),
}
jma_note = {
    "summer_tmax_mean_c": "JMA Tokyo", "summer_tmin_mean_c": "JMA Tokyo",
    "hot_days_ge33": "JMA Tokyo (≥35 °C)", "tropical_nights_ge25": "JMA Tokyo",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, (key, label, unit, color) in zip(axes.flat, specs):
    js = jma_series[key].values.astype(float)
    gj = fit_gam(jyears, js)
    ge = gams[key]
    ax.plot(years, heat[key].values, "o", color=color, ms=3, alpha=0.45, label="ERA5-Land annual")
    ax.plot(ge.x, ge.fitted, color=color, lw=2.3, label="ERA5-Land GAM")
    ax.plot(jyears, js, "s", color="gray", ms=3, alpha=0.4, label=f"{jma_note[key]} annual")
    ax.plot(gj.x, gj.fitted, color="black", lw=2.0, ls="--", label=f"{jma_note[key]} GAM")
    ax.set_xlabel("Year"); ax.set_ylabel(f"{label.split(' (')[0]} ({unit})")
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7, loc="upper left")
fig.suptitle("RQ1: ERA5-Land vs JMA Tokyo GAM smooths (trend-direction validation)", fontsize=12)
savefig(fig, 6, 1, "jma_validation_four")
plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\381338881.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Observation.** All four ERA5-Land smooths rise together with their JMA Tokyo counterparts, so the trends are corroborated by the station record. For all four the ERA5-Land Kanto area-mean sits *below* the Tokyo station, which is the expected direction: the reanalysis carries a cold bias (notebook 03), and a Kanto-wide average is cooler than a single dense-urban station whose night-time heat island and hot-day frequency both exceed the regional mean. The gap is widest for the counts, where the urban core crosses the thresholds far more often than the regional average. For hot days the comparison is ERA5-Land ≥33 °C against the station's standard ≥35 °C: both rise, so the adjusted grid count tracks the direction of the standard station count, though the urban core climbs more steeply. Direction, not level, is the valid takeaway; the point-versus-grid bias itself is quantified in notebook 03.

## 7. Model diagnostics

A GAM's confidence band and significance rest on its residual assumptions, and the two groups of series test those assumptions differently. The magnitudes (daily maximum and minimum temperature) are continuous and should approximately satisfy the Gaussian, constant-variance assumption. The counts (hot days, tropical nights) are bounded below by zero and grow from near-zero early in the record, so they are expected to show heteroscedasticity (variance growing with the mean) and heavier tails. All four are shown below, the two magnitudes first, then the two counts.

In [16]:
diag = [
    ("summer_tmax_mean_c",   "Mean daily maximum temperature (magnitude)", "tab:red",    1, "residuals_magnitude_tmax"),
    ("summer_tmin_mean_c",   "Mean daily minimum temperature (magnitude)", "tab:purple", 2, "residuals_magnitude_tmin"),
    ("hot_days_ge33",        "Hot days ≥33 °C (count)",               "tab:orange", 3, "residuals_count_hot_days"),
    ("tropical_nights_ge25", "Tropical nights ≥25 °C (count)",        "tab:brown",  4, "residuals_count_tropical_nights"),
]
for key, label, color, n, fname in diag:
    fig = residual_diagnostics(gams[key], years, heat[key].values, label, color=color)
    savefig(fig, 7, n, fname)
    plt.show()

C:\Users\dylan\AppData\Local\Temp\ipykernel_31564\3652577768.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Observation.** The four residual sets divide exactly along the magnitude/count line, which confirms the grouping is more than a labelling convenience. The two magnitudes are close to the Gaussian ideal: for both the mean daily maximum and the mean daily minimum the residuals show roughly constant spread against the fitted values, track the Q-Q line through the bulk with only mild departures in the lower tail (a few cool outlier years), and carry no trend or obvious autocorrelation against year. The two counts behave as count data should: for both hot days and tropical nights the residuals fan out as the fitted count grows (heteroscedasticity, expected for a variable bounded at zero) and the Q-Q tails are noticeably heavy. The practical consequence follows the same split: the analytic confidence band is trustworthy for the two magnitudes, and is best read as somewhat optimistic where the counts are high, which is why the Section 3 bootstrap cross-check matters more for the count series.

## 8. Synthesis

GAM summary statistics for the four series: model fit and shape (EDoF, deviance explained, selected λ, AIC) and `rate_sig_from`, the onset of the sustained significant-positive rate. These are trend descriptors, since the ERA5-Land counts are trend-only.

In [17]:
rows = []
for key, label in [
    ("summer_tmax_mean_c",   "Daily maximum temperature"),
    ("summer_tmin_mean_c",   "Daily minimum temperature"),
    ("hot_days_ge33",        "Hot days ≥33 °C"),
    ("tropical_nights_ge25", "Tropical nights ≥25 °C"),
]:
    g = gams[key]; r = rates[key]
    rows.append({"index": label, "edof": g.edof, "deviance_explained": g.deviance_explained,
                 "lam": g.lam, "aic": g.aic, "rate_sig_from": r["rate_sig_from"]})
gam_summary = pd.DataFrame(rows)
gam_summary.to_csv(PROCESSED_DIR / "rq1_gam_summary_v1.1.csv", index=False)
gam_summary.round(4)

,index,edof,deviance_explained,lam,aic,rate_sig_from
0,Daily maximum temperature,2.0356,0.3394,1000.0000,127.0425,1980.0000
1,Daily minimum temperature,3.7693,0.5003,3.1623,99.5676,2006.7538
2,Hot days ≥33 °C,2.0356,0.1944,1000.0000,227.1124,1980.0000
3,Tropical nights ≥25 °C,4.5247,0.6190,1.0000,262.4590,2007.1960


**RQ1 answer.** Over 1980–2024 every measure of Kanto summer heat has risen, but along two clearly different trajectories that divide by time of day, not by type of index. The daytime pair rise steadily: the mean daily maximum warms **approximately linearly** at about 0.05 °C per year (0.5 °C per decade), and the hot-day count (≥33 °C) rises with it, roughly linearly, from near-absence in the 1980s to around six days per year. The overnight pair **accelerate**: the mean daily minimum's rate of change passes through a lull around 2001 and then climbs, becoming sustained-significantly positive from about 2007 and reaching roughly 0.14 °C per year at the end of the record, and the tropical-night count (≥25 °C) traces the same shape with the same onset year, near-flat through the 1990s and then steepening to a growth of nearly two nights per year by 2024.

The count's sharper relative rise on top of the mean's acceleration is the expected geometry of threshold exceedance. The mean is a linear average over space and season, so it registers the underlying warming modestly; the count applies a fixed 25 °C threshold at each cell and day *before* averaging, so as the overnight temperature distribution shifts upward, ever more of it crosses the threshold and the exceedance count responds convexly. The two overnight indices are therefore one signal read through two functionals: an accelerating overnight warming, registered modestly in the mean and amplified in the count, with the spatial spread of warm-night areas seen in the exploratory maps contributing to the amplification. The comparatively flat hot-day count reflects a threshold sitting further out in the daytime tail and a far sparser series in which curvature cannot be resolved.

The answer to RQ1 is therefore that extreme-heat frequency has risen with a marked **diurnal asymmetry**: daytime heat (the mean maximum and the hot-day count) has increased at a steady pace across the whole record, while overnight heat (the mean minimum and the tropical-night count) has accelerated since the mid-2000s and is rising fastest at the record's end. (The 2020–2024 values sit in a short, outlier-heavy five-year window and should be read with that caveat.) Rank-based cross-checks in a companion notebook agree in direction and significance.

### A hypothesis on mechanism (for future work)

Why is the accelerating signal specifically an *overnight* one? The spatial maps in notebook 01 show the tropical-night increase concentrated along the coasts and, tellingly, around Lake Biwa, an inland cell that behaves like a coastal one. A guarded, thermodynamically-motivated hypothesis follows. Large water bodies have high heat capacity and thermal inertia, so they cool slowly overnight and hold the local daily minimum up; as the surrounding water warms (the ocean, and inland Lake Biwa), that elevated overnight floor is pushed across the 25 °C threshold more often, preferentially in near-water cells. This is *consistent with* the documented warming of the seas around Japan, and it acts on minima (nights) more than maxima (days), which fits the overnight measures, the minimum-temperature mean and the tropical-night count, being the ones that accelerate.

This is a hypothesis, not a result, and two limits keep it guarded. First, the coastal concentration is confounded with the urban heat island, since the largest cities sit on the bays; Lake Biwa is the natural discriminator (a large water body with far less urbanisation) but not a clean control. Second, ERA5-Land is a land-only reanalysis, so attributing the pattern to sea-surface warming would require SST data beyond this notebook's scope. Testing this, by bringing in SST fields and separating the water-proximity effect from urbanisation, is left as future work.